# Replication quantized pass - gemma-2-2b-it (NF4)

NF4 on **google/gemma-2-2b-it** @ `299a8560bedf`, on the
languages that cleared the FP16 floor gate. FP16 is NOT re-run; those cells
already exist from the gate run.

Tests pre-registered prediction 1: languages this model tokenizes cheaply
should show smaller FP16-to-NF4 degradation than Qwen showed for them.

**Settings: Accelerator `GPU T4 x2`, Internet `ON`.**

In [ ]:
# 1. Get the code.
REPO_URL = "https://github.com/fairuz-anadi/quantization.git"
REF      = "main"

import os, subprocess, sys
SRC = "/kaggle/working/quantlang"
if not os.path.exists(SRC):
    subprocess.run(["git", "clone", REPO_URL, SRC], check=True)
    subprocess.run(["git", "-C", SRC, "checkout", "--quiet", REF], check=True)
print(subprocess.run(["git", "-C", SRC, "rev-parse", "HEAD"],
                     capture_output=True, text=True).stdout.strip())
os.chdir(SRC); sys.path.insert(0, SRC)

In [ ]:
!pip install -q -U "transformers>=4.45" "bitsandbytes>=0.43" "peft>=0.13" accelerate datasets pyyaml
!pip uninstall -q -y torchao

In [ ]:
import subprocess, sys
def gate(*cmd):
    print('$', ' '.join(cmd), flush=True)
    if subprocess.run([sys.executable, *cmd]).returncode != 0:
        raise SystemExit('GATE FAILED: ' + ' '.join(cmd) + '. Stop here -- '
                         'the design is not adjusted to make a check pass.')

gate("scripts/probe_env.py", "--outdir", "/kaggle/working")

## P0 is still frozen, and the suite still passes

In [ ]:
import subprocess, sys
def gate(*cmd):
    print('$', ' '.join(cmd), flush=True)
    if subprocess.run([sys.executable, *cmd]).returncode != 0:
        raise SystemExit('GATE FAILED: ' + ' '.join(cmd) + '. Stop here -- '
                         'the design is not adjusted to make a check pass.')

gate("scripts/freeze_p0.py")
gate("-m", "pytest", "-q")

## NF4

Same evaluator, same frozen 900-item manifest, same `letter_logit` scoring as
P0 and as the FP16 gate run. Languages: `eng_Latn`, `ben_Beng`, `sin_Sinh`, `asm_Beng`, `npi_Deva`.

In [ ]:
import subprocess, sys
def gate(*cmd):
    print('$', ' '.join(cmd), flush=True)
    if subprocess.run([sys.executable, *cmd]).returncode != 0:
        raise SystemExit('GATE FAILED: ' + ' '.join(cmd) + '. Stop here -- '
                         'the design is not adjusted to make a check pass.')

for prec in ['nf4']:
    gate("scripts/run_eval.py", "--precision", prec,
         "--langs",
         'eng_Latn',
         'ben_Beng',
         'sin_Sinh',
         'asm_Beng',
         'npi_Deva',
         "--model-alias", "gemma-2-2b-it",
         "--outdir", "/kaggle/working/rep/results",
         "--tag", "repq")

## Cells written

In [ ]:
import glob, json
rows = []
for p in sorted(glob.glob("/kaggle/working/rep/results/*.meta.json")):
    m = json.load(open(p, encoding="utf-8"))
    assert m["arm"] == "base", "a replication cell must be a base cell"
    assert m["n_items"] == 900, m["n_items"]
    rows.append(m)

print(f"{"lang":10}{"precision":16}{"acc":>8}{"items":>7}{"tok":>7}{"trunc":>7}")
for m in sorted(rows, key=lambda m: (m["lang"], m["precision"])):
    print(f"{m['lang']:10}{m['precision']:16}{m['accuracy']:8.4f}"
          f"{m['n_items']:7d}{m['median_input_tokens']:7.0f}{m['n_truncated']:7d}")
print("\nBring these back with the FP16 gate numbers for the paired analysis.")

In [ ]:
import os, shutil
KEEP = "/kaggle/working/repq_gemma-2-2b-it_keep"
os.makedirs(KEEP, exist_ok=True)
shutil.copytree("/kaggle/working/rep/results", KEEP + "/results", dirs_exist_ok=True)
shutil.make_archive("/kaggle/working/repq_gemma-2-2b-it", "zip", KEEP)
print(sorted(os.listdir("/kaggle/working")))